In [1]:
class Region:
    def __init__(self, name: str, priority: float, bits: int = 4):
        self.name = name
        self.priority = priority
        self.bits = bits
        self.binary_vars = [(self.name, k) for k in range(bits)]

    def allocation_terms(self):
        return [(bit, 2 ** k) for k, bit in enumerate(self.binary_vars)]

In [ ]:
def build_var_index(regions, slack_bit_count):
    var_index = {}
    idx = 0

    for region in regions:
        for bit in region.binary_vars:
            var_index[bit] = idx
            idx += 1

    for l in range(slack_bit_count):
        var_index[('s', l)] = idx
        idx += 1

    return var_index

In [ ]:
def add(u, v, coeff, Q):
    if u > v:
        u, v = v, u

    if u not in Q:
        Q[u] = {}
    if v not in Q[u]:
        Q[u][v] = 0.0

    Q[u][v] += coeff

In [ ]:
def add_priority_objective(Q, var_index, regions):
    for region in regions:
        for (bit_label, weight) in region.allocation_terms():
            idx = var_index[bit_label]
            coeff = -region.priority * weight
            add(idx, idx, coeff, Q)

In [ ]:
def add_capacity_penalty(Q, var_index, regions, C, slack_bits, P2):
    terms = []

    for region in regions:
        for (bit_label, weight) in region.allocation_terms():
            idx = var_index[bit_label]
            terms.append((idx, weight))

    for l in range(slack_bits):
        label = ("s", l)
        idx = var_index[label]
        terms.append((idx, 2**l))

    for i in range(len(terms)):
        u, wu = terms[i]
        add(u, u, P2 * (wu**2 - 2 * C * wu), Q)
        for j in range(i+1, len(terms)):
            v, wv = terms[j]
            add(u, v, 2 * P2 * wu * wv, Q)

In [ ]:
def add_fairness_penalty(Q, var_index, regions, lam_fair):
    n = len(regions)
    scale = lam_fair / (2 * n * n)

    for i in range(n):
        for j in range(i + 1, n):
            reg_i = regions[i]
            reg_j = regions[j]

            for reg in (reg_i, reg_j):
                terms = reg.allocation_terms()
                for a, (label_a, w_a) in enumerate(terms):
                    idx_a = var_index[label_a]
                    add(idx_a, idx_a, scale * (w_a ** 2), Q)
                    for (label_b, w_b) in terms[a + 1:]:
                        add(idx_a, var_index[label_b], 2 * scale * w_a * w_b, Q)

            for (label_i, w_i) in reg_i.allocation_terms():
                idx_i = var_index[label_i]
                for (label_j, w_j) in reg_j.allocation_terms():
                    idx_j = var_index[label_j]
                    add(idx_i, idx_j, -2 * scale * w_i * w_j, Q)

In [ ]:
import dimod

def build_water_qubo(regions, capacity, slack_bits, lam_cap, lam_fair):
    Q = {}  
    var_index = build_var_index(regions, slack_bits)

    add_priority_objective(Q, var_index, regions)
    add_capacity_penalty(Q, var_index, regions, capacity, slack_bits, lam_cap)
    add_fairness_penalty(Q, var_index, regions, lam_fair)

    Q_flat = {(u, v): c for u, row in Q.items() for v, c in row.items()}
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_flat)

    return bqm, var_index

In [9]:
import neal

def solve_with_annealing(bqm, num_reads=50, num_sweeps=5000):
    sampler = neal.SimulatedAnnealingSampler()
    result = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps)
    best_sample = result.first.sample
    return best_sample

In [10]:
def decode_solution(sample, var_index, regions):
    allocation = {}
    for region in regions:
        value = 0
        for (label, weight) in region.allocation_terms():
            bit_idx = var_index[label]
            bit_val = sample.get(bit_idx, 0)
            value += weight * bit_val
        allocation[region.name] = value
    return allocation

In [ ]:
def check_feasibility(sample, var_index, regions, slack_bit_count, capacity):
    total_alloc = 0
    for region in regions:
        for (label, weight) in region.allocation_terms():
            bit_idx = var_index[label]
            bit_val = sample.get(bit_idx, 0)
            total_alloc += weight * bit_val

    slack_val = 0
    for l in range(slack_bit_count):
        bit_idx = var_index[('s', l)]
        bit_val = sample.get(bit_idx, 0)
        slack_val += (2 ** l) * bit_val

    feasible = (total_alloc <= capacity)

    print(f"Total Allocation: {total_alloc}")
    print(f"Slack Used: {slack_val}")
    print(f"Capacity: {capacity}")
    print(f"Feasible: {feasible}")

    return feasible